# Empirical Analysis: Law & Crime Topic Prevalence Under Cross-National and Document-Type Shift

This notebook reproduces the ACS analysis structure for an LLM-based measurement task.
We use Llama 3.3 70B Instruct (4-bit quantized) to classify parliamentary and media texts
from the Comparative Agendas Project (CAP) as related to Law & Crime (CAP major topic
code 12), calibrate with Isotonic Regression and MCGrad on a balanced multi-country
sample, then measure prevalence estimation bias across a shift gradient from no shift
to maximum shift (unseen document type).

**Sub-populations (6 total):**

| Country | Doc type | Language | N (approx) | Role |
|---------|----------|----------|------------|------|
| Denmark | Parl. questions | Danish | 15K | Calibration + test |
| Spain | Oral questions | Spanish | 15K | Calibration + test |
| US | Congressional bills | English | 15K | Calibration + test |
| Belgium | Newspaper | Dutch | 15K | Calibration + test |
| Spain | Media (El Pais + El Mundo) | Spanish | 30K | OOD target |
| Belgium | TV news | Dutch | 15K | OOD target |

**Calibration design:** Balanced sample from 4 sub-populations (Denmark questions,
Spain questions, US bills, Belgium newspaper) gives MCGrad variation in country,
language, doc_type, decade, and party. All 4 countries and all 4 languages are
represented in calibration. OOD targets (Spain media, Belgium TV) share country
and language with calibration but introduce an unseen document type — mirroring
the realistic scenario of validating on one document type and applying to another.

In [ ]:
import ctypes
import platform
if platform.system() == 'Linux':
    ctypes.cdll.LoadLibrary('/usr/lib64/libgomp.so.1')

import sys
import os

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

from mcgrad import methods as mcgrad_methods

sys.path.insert(0, "..")
from plot_config import METHOD_COLORS

from helpers import (
    calibrate_threshold_youden,
    estimate_classifier_error_rates,
    sld_estimate,
    pacc_estimate,
    compute_rogan_gladen_estimate,
    compute_all_prevalence_estimates,
    compute_bias_table,
    resample_with_party_shift,
    resample_with_doctype_shift,
    resample_with_country_shift,
    compute_bootstrap_rmse,
    compute_bootstrap_rmse_shifted,
)

os.makedirs('../paper/images', exist_ok=True)

plt.rcParams.update({
    'font.size': 11,
    'axes.titlesize': 13,
    'axes.labelsize': 11,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
    'legend.fontsize': 8,
    'figure.dpi': 150,
})

## 1. Data Loading

Load the full 105K sample (15K per sub-population across 7 sub-populations
in 4 countries) and join with Llama 3.3 70B inference scores.

In [ ]:
DATA_DIR = os.path.join('data')
LABEL_COLUMN = 'law_crime'
SCORE_COLUMN = 'llm_score'

# Load full sample and scores
data_df = pd.read_csv(os.path.join(DATA_DIR, 'full_sample.csv'))

# --- Score source selection ---
# Verbalized 2-stage scores (0-100 confidence, 2-stage dialogue with nudge)
# Scores are in [0.01, 0.99] — no squashing needed for MCGrad's logit transform.
SCORE_SOURCE = 'verbalized-2stage'

if SCORE_SOURCE == 'logprob':
    scores_df = pd.read_csv(
        os.path.join(DATA_DIR, 'inference_output', 'llama-70b', 'full_scores.csv')
    )
    NEEDS_SQUASHING = True
elif SCORE_SOURCE == 'verbalized-2stage':
    scores_df = pd.read_csv(
        os.path.join(DATA_DIR, 'inference_output', 'llama-70b-verbalized-2stage', 'full_codebook.csv')
    )
    NEEDS_SQUASHING = False
else:
    raise ValueError(f"Unknown score source: {SCORE_SOURCE}")

print(f"Score source: {SCORE_SOURCE}")
print(f"  Loaded {len(scores_df):,} scores")

# Join scores positionally (same row order, verified during data prep)
data_df[SCORE_COLUMN] = scores_df['score'].astype(float)

# Squashing: only needed for logprob scores where MCGrad's logit transform
# maps extreme values (exact 0/1) to +/-inf.
# Verbalized 2-stage scores are in [0.01, 0.99], so no squashing needed.
EPSILON = 0.05
SQUASHED_COL = 'llm_score_squashed'
if NEEDS_SQUASHING:
    data_df[SQUASHED_COL] = EPSILON + (1 - 2 * EPSILON) * data_df[SCORE_COLUMN]
    print(f"  Squashing applied: [{EPSILON}, {1-EPSILON}]")
else:
    data_df[SQUASHED_COL] = data_df[SCORE_COLUMN]
    print(f"  No squashing needed (scores in [{data_df[SCORE_COLUMN].min():.2f}, {data_df[SCORE_COLUMN].max():.2f}])")

# Create subpop key matching the 7 sub-population structure
SUBPOP_MAP = {
    ('Denmark', 'parliamentary_question'): 'denmark_questions',
    ('Spain', 'parliamentary_question'): 'spain_questions',
    ('Spain', 'media'): 'spain_media',
    ('United States', 'bill'): 'us_bills',
    ('Belgium', 'tv_news'): 'belgium_tv',
    ('Belgium', 'newspaper'): 'belgium_newspaper',
}
data_df['subpop'] = data_df.apply(
    lambda r: SUBPOP_MAP.get((r['country'], r['doc_type']), 'unknown'), axis=1
)

# Split into sub-population DataFrames
subpops = {}
for key in SUBPOP_MAP.values():
    sub = data_df[data_df['subpop'] == key].copy()
    subpops[key] = sub

print(f"\nSub-population summary:")
print(f"{'Key':<25} {'N':>7} {'Prevalence':>10} {'Mean Score':>10} {'Score range':>14}")
print("-" * 70)
for key, df in subpops.items():
    print(f"{key:<25} {len(df):>7,} {df[LABEL_COLUMN].mean():>9.1%} "
          f"{df[SCORE_COLUMN].mean():>10.3f} [{df[SCORE_COLUMN].min():.2f}, {df[SCORE_COLUMN].max():.2f}]")
print(f"\nTotal: {len(data_df):,} documents")

In [ ]:
from sklearn.metrics import roc_auc_score

print("=== LLM discriminative performance (AUC) by sub-population ===")
for key, df in subpops.items():
    auc = roc_auc_score(df[LABEL_COLUMN], df[SCORE_COLUMN])
    pos_mean = df.loc[df[LABEL_COLUMN] == 1, SCORE_COLUMN].mean()
    neg_mean = df.loc[df[LABEL_COLUMN] == 0, SCORE_COLUMN].mean()
    print(f"  {key:<25} AUC={auc:.3f}  pos_mean={pos_mean:.3f}  neg_mean={neg_mean:.3f}")

print("\n=== Year range by sub-population ===")
for key, df in subpops.items():
    print(f"  {key}: {df['year'].min()}--{df['year'].max()}")

print("\n=== Party coverage ===")
for key, df in subpops.items():
    if 'party' in df.columns and df['party'].notna().any():
        n_parties = df['party'].nunique()
        coverage = df['party'].notna().mean()
        print(f"  {key}: {n_parties} parties, {coverage:.0%} coverage")
    else:
        print(f"  {key}: no party data")

### Load LLM Embeddings

Load Llama 3.3 70B last-token hidden states (8192-dim) for all 105K documents.
If not available locally, download from manifold.

In [ ]:
import subprocess

EMB_DIR = os.path.join(DATA_DIR, 'inference_output', 'llama-70b', 'embeddings')
EMB_PATH = os.path.join(EMB_DIR, 'embeddings.npy')
EMB_IDS_PATH = os.path.join(EMB_DIR, 'embedding_ids.csv')
MANIFOLD_PATH = 'multicalibration/tree/mc_measurement_paper/cap_embeddings/llama-3.3-70b'

HAS_EMBEDDINGS = False

if not os.path.exists(EMB_PATH):
    print(f"Embeddings not found locally at {EMB_PATH}")
    try:
        print(f"Attempting download from manifold: {MANIFOLD_PATH} ...")
        os.makedirs(EMB_DIR, exist_ok=True)
        result = subprocess.run(
            ['manifold', '--prod-use-cython-client', 'getr',
             MANIFOLD_PATH, EMB_DIR, '--threads', '20', '--jobs', '10'],
            capture_output=True, text=True, timeout=120,
        )
        if result.returncode != 0:
            print(f"Manifold download failed (exit {result.returncode}). "
                  "Skipping embedding-based methods.")
        else:
            print("Download complete.")
            HAS_EMBEDDINGS = True
    except (FileNotFoundError, subprocess.TimeoutExpired) as e:
        print(f"Manifold not available ({type(e).__name__}). "
              "Skipping embedding-based methods.")
else:
    HAS_EMBEDDINGS = True

if HAS_EMBEDDINGS:
    # Load embeddings and ID mapping
    embeddings_raw = np.load(EMB_PATH)
    emb_ids_df = pd.read_csv(EMB_IDS_PATH)

    print(f"Embeddings shape: {embeddings_raw.shape}  dtype: {embeddings_raw.dtype}")
    print(f"Embedding IDs: {len(emb_ids_df):,} rows")

    assert len(emb_ids_df) == embeddings_raw.shape[0], \
        f"ID count ({len(emb_ids_df)}) != embedding rows ({embeddings_raw.shape[0]})"

    # Build ID-to-row-index mapping for aligning with data_df
    emb_ids_df['id'] = emb_ids_df['id'].astype(str)
    emb_id_to_idx = dict(zip(emb_ids_df['id'], emb_ids_df['row_idx']))

    # Verify alignment with data_df
    data_ids = data_df['id'].astype(str)
    n_matched = data_ids.isin(emb_id_to_idx).sum()
    print(f"Matched {n_matched:,} / {len(data_df):,} documents with embeddings")
else:
    print("\n*** Running without embeddings. MCGrad + Emb. method will be skipped. ***")

### Score Distribution Diagnostic

The LLM produces log-probability-based scores via P(Yes) / (P(Yes) + P(No)).
These are not calibrated posteriors — they tend to be **bimodal** (near 0 or near 1),
with a substantial fraction of negatives receiving very high scores. This has
important implications for all downstream methods.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
subpop_order = ['denmark_questions', 'spain_questions', 'us_bills',
                'belgium_newspaper', 'spain_media', 'belgium_tv']
titles = ['Denmark Questions\n(calibration)', 'Spain Questions\n(calibration)',
          'US Bills\n(calibration)', 'Belgium Newspaper\n(calibration)',
          'Spain Media\n(OOD)', 'Belgium TV\n(OOD)']

for ax, key, title in zip(axes.flat, subpop_order, titles):
    df = subpops[key]
    neg_scores = df.loc[df[LABEL_COLUMN] == 0, SCORE_COLUMN]
    pos_scores = df.loc[df[LABEL_COLUMN] == 1, SCORE_COLUMN]

    ax.hist(neg_scores, bins=50, alpha=0.6, color='steelblue',
            label=f'Y=0 (n={len(neg_scores):,})', density=True)
    ax.hist(pos_scores, bins=50, alpha=0.6, color='coral',
            label=f'Y=1 (n={len(pos_scores):,})', density=True)
    ax.set_title(title, fontsize=10)
    ax.set_xlabel('LLM Score')
    ax.legend(fontsize=6)

    # Show fraction of extreme scores
    high_neg = (neg_scores > 0.99).mean()
    high_pos = (pos_scores > 0.99).mean()
    low_neg = (neg_scores < 0.01).mean()
    ax.text(0.5, 0.95,
            f'Y=0 > 0.99: {high_neg:.1%}\nY=1 > 0.99: {high_pos:.1%}\nY=0 < 0.01: {low_neg:.1%}',
            transform=ax.transAxes, fontsize=6, va='top', ha='center',
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

fig.suptitle('LLM Score Distribution by Label and Sub-population', fontsize=13)
fig.tight_layout()
fig.savefig(
    '../paper/images/figure_cap_score_distribution.png',
    dpi=300, bbox_inches='tight',
)
plt.show()

# Aggregate extreme value statistics
print("\n=== Extreme value analysis ===")
print(f"{'Subpop':<25} {'%neg>0.99':>10} {'%neg<0.01':>10} {'%pos>0.99':>10} {'%pos<0.01':>10}")
print("-" * 70)
for key in subpop_order:
    df = subpops[key]
    neg = df.loc[df[LABEL_COLUMN] == 0, SCORE_COLUMN]
    pos = df.loc[df[LABEL_COLUMN] == 1, SCORE_COLUMN]
    print(f"{key:<25} {(neg > 0.99).mean():>9.1%} {(neg < 0.01).mean():>9.1%} "
          f"{(pos > 0.99).mean():>9.1%} {(pos < 0.01).mean():>9.1%}")

## 2. Calibration Setup

Calibrate on a **balanced sample from 4 sub-populations** to give MCGrad
variation in country, language, doc_type, decade, and party:
- Denmark questions (~15K)
- Spain questions (~15K)
- US bills (~15K)
- Belgium newspaper (~15K)

Total calibration: ~40K (10K per sub-population, ~67% split). The remaining data
from these 4 sub-populations forms the in-distribution test set (~20K).

All 4 countries and all 4 languages are represented in calibration.

**OOD targets** (share country and language with calibration, unseen doc_type):
- Spain media (El Pais + El Mundo) -- same country/language, new doc type
- Belgium TV news -- same country/language, new doc type

In [ ]:
# Build calibration set: balanced sample from 4 sub-populations
CAL_SUBPOPS = ['denmark_questions', 'spain_questions', 'us_bills', 'belgium_newspaper']

calibration_parts = []
test_parts = []

for key in CAL_SUBPOPS:
    df = subpops[key].copy()
    # Target ~10K per subpop (~67% calibration) to give MCGrad more data
    # for learning feature-conditional corrections
    cal_frac = min(10_000 / len(df), 0.67)
    cal_part, test_part = train_test_split(
        df,
        test_size=1.0 - cal_frac,
        random_state=42,
        stratify=df[LABEL_COLUMN],
    )
    calibration_parts.append(cal_part)
    test_parts.append(test_part)
    print(f"  {key}: {len(cal_part):,} calibration, {len(test_part):,} test")

calibration_df = pd.concat(calibration_parts, ignore_index=True)
test_df = pd.concat(test_parts, ignore_index=True)

print(f"\nCalibration set: {len(calibration_df):,} samples")
print(f"  Law & Crime prevalence: {calibration_df[LABEL_COLUMN].mean():.1%}")
print(f"  By country: {calibration_df['country'].value_counts().to_dict()}")
print(f"  By doc_type: {calibration_df['doc_type'].value_counts().to_dict()}")

print(f"\nIn-distribution test set: {len(test_df):,} samples")
print(f"  Law & Crime prevalence: {test_df[LABEL_COLUMN].mean():.1%}")

In [ ]:
# --- Feature preparation ---
# Create decade feature and fill missing party for all DataFrames

all_dfs_to_prep = [calibration_df, test_df] + [
    subpops[k] for k in subpops if k not in CAL_SUBPOPS
]
for df in all_dfs_to_prep:
    df['decade'] = (df['year'] // 10) * 10
    # Fill missing party with 'unknown' for MCGrad compatibility
    if 'party' in df.columns:
        df['party'] = df['party'].fillna('unknown')
    else:
        df['party'] = 'unknown'

### PCA Embedding Features

Reduce the 8192-dim Llama 3.3 70B last-token hidden states to 500 PCA components.
PCA is fitted on the calibration set only to avoid data leakage. These components
are added as numerical features for a second MCGrad model ("MCGrad + Emb.")
to test whether text-level embedding information improves OOD prevalence estimation.

In [ ]:
if HAS_EMBEDDINGS:
    from sklearn.decomposition import PCA

    N_PCA_COMPONENTS = 500

    # Fit PCA on calibration set embeddings only
    cal_emb_indices = calibration_df['id'].astype(str).map(emb_id_to_idx).values
    cal_embeddings = embeddings_raw[cal_emb_indices].astype(np.float32)

    pca = PCA(n_components=N_PCA_COMPONENTS, random_state=42)
    pca.fit(cal_embeddings)

    explained_var = pca.explained_variance_ratio_.sum()
    print(f"PCA: {N_PCA_COMPONENTS} components explain {explained_var:.1%} of variance")
    print(f"  Top 10 components: {pca.explained_variance_ratio_[:10].sum():.1%}")
    print(f"  Top 50 components: {pca.explained_variance_ratio_[:50].sum():.1%}")
    print(f"  Top 100 components: {pca.explained_variance_ratio_[:100].sum():.1%}")

    # Add PCA columns to all evaluation DataFrames
    PCA_COLS = [f'pca_{i}' for i in range(N_PCA_COMPONENTS)]
    OOD_KEYS = [k for k in subpops if k not in CAL_SUBPOPS]

    all_eval_dfs = [
        ('calibration', calibration_df),
        ('test', test_df),
    ] + [(k, subpops[k]) for k in OOD_KEYS]

    for label, df in all_eval_dfs:
        emb_indices = df['id'].astype(str).map(emb_id_to_idx).values
        raw_emb = embeddings_raw[emb_indices].astype(np.float32)
        pca_emb = pca.transform(raw_emb)
        for i in range(N_PCA_COMPONENTS):
            df[f'pca_{i}'] = pca_emb[:, i]
        print(f"  {label}: added {N_PCA_COMPONENTS} PCA columns ({len(df):,} rows)")

    del cal_embeddings  # free memory
else:
    PCA_COLS = []
    print("Skipping PCA (no embeddings available)")

In [ ]:
# --- Fit calibration methods ---

# 1. Isotonic Regression (global calibration) — fitted on raw scores
isotonic_reg = mcgrad_methods.IsotonicRegression().fit(
    calibration_df,
    SCORE_COLUMN,
    LABEL_COLUMN,
)
print("Isotonic regression fitted (on raw scores)")

# 2. MCGrad (multicalibration) — fitted on squashed scores
# Squashing maps [0,1] -> [0.05, 0.95] so MCGrad's logit transform
# doesn't clip extreme values.
CATEGORICAL_SEGMENT_FEATURES = ['doc_type', 'country', 'party']
NUMERICAL_SEGMENT_FEATURES = ['decade']

mcgrad = mcgrad_methods.MCGrad(save_training_performance=True)
mcgrad = mcgrad.fit(
    calibration_df,
    SQUASHED_COL,
    LABEL_COLUMN,
    categorical_feature_column_names=CATEGORICAL_SEGMENT_FEATURES,
    numerical_feature_column_names=NUMERICAL_SEGMENT_FEATURES,
)
print("MCGrad fitted (on squashed scores)")

In [ ]:
# 3. MCGrad + Embeddings — fitted on squashed scores with PCA embedding features
MCGRAD_EMB_COL = 'mcgrad_emb_prediction'

if HAS_EMBEDDINGS:
    EMB_CATEGORICAL_FEATURES = ['doc_type', 'country', 'party']
    EMB_NUMERICAL_FEATURES = ['decade'] + PCA_COLS

    mcgrad_emb = mcgrad_methods.MCGrad(save_training_performance=True)
    mcgrad_emb = mcgrad_emb.fit(
        calibration_df,
        SQUASHED_COL,
        LABEL_COLUMN,
        categorical_feature_column_names=EMB_CATEGORICAL_FEATURES,
        numerical_feature_column_names=EMB_NUMERICAL_FEATURES,
    )
    print(f"MCGrad + Embeddings fitted "
          f"({len(EMB_CATEGORICAL_FEATURES)} categorical + "
          f"{len(EMB_NUMERICAL_FEATURES)} numerical features)")
else:
    mcgrad_emb = None
    print("Skipping MCGrad + Embeddings (no embeddings available)")

In [ ]:
# Generate calibrated predictions for test set and all OOD sub-populations
IR_COL = 'isotonic_prediction'
MCGRAD_COL = 'mcgrad_prediction'

# OOD sub-populations (not part of calibration split)
OOD_KEYS = [k for k in subpops if k not in CAL_SUBPOPS]

eval_dfs = [test_df] + [subpops[k] for k in OOD_KEYS]

for df in eval_dfs:
    # Isotonic regression predictions (from raw scores)
    df[IR_COL] = isotonic_reg.predict(df, SCORE_COLUMN)

    # MCGrad predictions (from squashed scores)
    df[MCGRAD_COL] = mcgrad.predict(
        df=df,
        prediction_column_name=SQUASHED_COL,
        categorical_feature_column_names=CATEGORICAL_SEGMENT_FEATURES,
        numerical_feature_column_names=NUMERICAL_SEGMENT_FEATURES,
    )

    # MCGrad + Embeddings predictions (from squashed scores + PCA features)
    if HAS_EMBEDDINGS and mcgrad_emb is not None:
        EMB_CATEGORICAL_FEATURES = ['doc_type', 'country', 'party']
        EMB_NUMERICAL_FEATURES = ['decade'] + PCA_COLS
        df[MCGRAD_EMB_COL] = mcgrad_emb.predict(
            df=df,
            prediction_column_name=SQUASHED_COL,
            categorical_feature_column_names=EMB_CATEGORICAL_FEATURES,
            numerical_feature_column_names=EMB_NUMERICAL_FEATURES,
        )

print("Calibrated predictions generated for all evaluation datasets")
if not HAS_EMBEDDINGS:
    print("  (MCGrad + Emb. skipped — no embeddings)")

### MCGrad Fit Evaluation

Learning curve showing training and validation loss across MCGrad boosting rounds,
followed by discriminative performance (PRAUC, ROCAUC, log-loss) and calibration
metrics (ECCE, MCE) comparing raw LLM scores, isotonic regression, and MCGrad.

In [ ]:
# === MCGrad Fit Evaluation ===
from mcgrad import metrics as mcgrad_metrics, plotting as mcgrad_plotting
from sklearn.metrics import average_precision_score, roc_auc_score, log_loss

# Learning curve(s)
mcgrad_plotting.plot_learning_curve(mcgrad, show_all=True).update_layout(
    width=700, height=500, title="MCGrad Learning Curve"
).show()

if HAS_EMBEDDINGS and mcgrad_emb is not None:
    mcgrad_plotting.plot_learning_curve(mcgrad_emb, show_all=True).update_layout(
        width=700, height=500, title="MCGrad + Emb. Learning Curve"
    ).show()

# Discriminative and calibration metrics
eval_metric_fns = {
    "PRAUC": average_precision_score,
    "ROCAUC": roc_auc_score,
    "Log-loss": log_loss,
}

score_columns = {
    "Raw LLM Scores": SCORE_COLUMN,
    "Isotonic Regression": IR_COL,
    "MCGrad": MCGRAD_COL,
}
if HAS_EMBEDDINGS and MCGRAD_EMB_COL in test_df.columns:
    score_columns["MCGrad + Emb."] = MCGRAD_EMB_COL

eval_results = {}
for name, col in score_columns.items():
    row = {
        metric_name: metric_func(test_df[LABEL_COLUMN].values, test_df[col].values)
        for metric_name, metric_func in eval_metric_fns.items()
    }
    ecce_val = mcgrad_metrics.ecce(test_df[LABEL_COLUMN].values, test_df[col].values)
    ecce_sig = mcgrad_metrics.ecce_sigma(test_df[LABEL_COLUMN].values, test_df[col].values)
    mce_obj = mcgrad_metrics.MulticalibrationError(
        df=test_df,
        label_column=LABEL_COLUMN,
        score_column=col,
        categorical_segment_columns=CATEGORICAL_SEGMENT_FEATURES,
        numerical_segment_columns=NUMERICAL_SEGMENT_FEATURES,
    )
    row["ECCE"] = ecce_val
    row["ECCE σ"] = ecce_sig
    row["MCE"] = mce_obj.mce
    row["MCE σ"] = mce_obj.mce_sigma
    eval_results[name] = row

pd.DataFrame(eval_results).T.round(4)

In [ ]:
# --- Calibration parameters for quantification methods ---

THRESHOLD = calibrate_threshold_youden(
    labels=calibration_df[LABEL_COLUMN],
    predictions=calibration_df[SCORE_COLUMN],
)
print(f"Calibrated threshold (Youden's J): {THRESHOLD:.4f}")

calibration_tpr, calibration_fpr = estimate_classifier_error_rates(
    labels=calibration_df[LABEL_COLUMN],
    predictions=calibration_df[SCORE_COLUMN],
    threshold=THRESHOLD,
)
print(f"\nCalibration set error rates at threshold={THRESHOLD:.4f}:")
print(f"  TPR (sensitivity): {calibration_tpr:.4f}")
print(f"  FPR (1-specificity): {calibration_fpr:.4f}")

# PACC parameters
PACC_POS_MEAN = calibration_df[calibration_df[LABEL_COLUMN] == 1][SCORE_COLUMN].mean()
PACC_NEG_MEAN = calibration_df[calibration_df[LABEL_COLUMN] == 0][SCORE_COLUMN].mean()
SOURCE_PREVALENCE = calibration_df[LABEL_COLUMN].mean()

print(f"\nPACC parameters (soft-score Rogan-Gladen):")
print(f"  E[h(X)|Y=1]: {PACC_POS_MEAN:.4f}")
print(f"  E[h(X)|Y=0]: {PACC_NEG_MEAN:.4f}")
print(f"\nSource prevalence for SLD: {SOURCE_PREVALENCE:.4f}")

# Verify
apparent_prev = (calibration_df[SCORE_COLUMN] >= THRESHOLD).mean()
true_prev = calibration_df[LABEL_COLUMN].mean()
print(f"\nVerification:")
print(f"  True prevalence in calibration set: {true_prev:.4f}")
print(f"  Apparent prevalence at threshold: {apparent_prev:.4f}")

## 3. Shift Gradient Construction

We construct a gradient of increasing distributional shift:

| Scenario | Source | Shift type |
|----------|--------|------------|
| Baseline | Balanced test split (same composition as calibration) | None |
| Country shift | Test split resampled to overweight Belgium | Within-calibration |
| Doc-type shift | Test split resampled to overweight US bills | Within-calibration |
| Party shift | Test split resampled to overweight one party | Within-calibration |
| OOD: Spain media | Spain media (El Pais + El Mundo) | Same country/lang, new doc type |
| OOD: Belgium TV | Belgium TV news | Same country/lang, new doc type |

In [ ]:
N_SAMPLES = 20_000

scenarios = {
    'Baseline\n(balanced test)': {
        'df': test_df.sample(
            n=min(N_SAMPLES, len(test_df)),
            replace=True, random_state=42),
        'shift_type': 'none',
    },
    'Country shift\n(overweight Belgium)': {
        'df': resample_with_country_shift(
            test_df, target_country='Belgium',
            n_samples=N_SAMPLES, random_state=42),
        'shift_type': 'within-calibration',
    },
    'Doc-type shift\n(overweight bills)': {
        'df': resample_with_doctype_shift(
            test_df, target_doctype='bill',
            n_samples=N_SAMPLES, random_state=42),
        'shift_type': 'within-calibration',
    },
    'Party shift\n(right-heavy)': {
        'df': resample_with_party_shift(
            test_df, label_column=LABEL_COLUMN, shift='right_heavy',
            n_samples=N_SAMPLES, random_state=42),
        'shift_type': 'within-calibration',
    },
    'Spain media\n(OOD doc type)': {
        'df': subpops['spain_media'].sample(
            n=min(N_SAMPLES, len(subpops['spain_media'])),
            replace=True, random_state=42),
        'shift_type': 'OOD (new doc type)',
    },
    'Belgium TV\n(OOD doc type)': {
        'df': subpops['belgium_tv'].sample(
            n=min(N_SAMPLES, len(subpops['belgium_tv'])),
            replace=True, random_state=42),
        'shift_type': 'OOD (new doc type)',
    },
}

print("Shift gradient scenarios:")
print(f"{'Scenario':<35} {'N':>7}  {'True prev':>10}  {'Shift type'}")
print("-" * 80)
for label, info in scenarios.items():
    df = info['df']
    clean_label = label.replace('\n', ' ')
    print(f"{clean_label:<35} {len(df):>7,}  "
          f"{df[LABEL_COLUMN].mean():>9.1%}  {info['shift_type']}")

# --- Calibration parameters for quantification methods ---
from sklearn.metrics import roc_curve


def calibrate_threshold_youden(labels, predictions):
    """Find threshold that maximizes Youden's J = TPR - FPR."""
    fpr_arr, tpr_arr, thresholds = roc_curve(labels, predictions)
    j_scores = tpr_arr - fpr_arr
    best_idx = np.argmax(j_scores)
    return float(thresholds[best_idx])


def estimate_classifier_error_rates(labels, predictions, threshold):
    """Estimate TPR and FPR from calibration data at given threshold."""
    binary_preds = (predictions >= threshold).astype(int)
    labels_arr = labels.astype(int)
    tp = ((binary_preds == 1) & (labels_arr == 1)).sum()
    fp = ((binary_preds == 1) & (labels_arr == 0)).sum()
    tn = ((binary_preds == 0) & (labels_arr == 0)).sum()
    fn = ((binary_preds == 0) & (labels_arr == 1)).sum()
    tpr = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    fpr = fp / (fp + tn) if (fp + tn) > 0 else 0.0
    return tpr, fpr


THRESHOLD = calibrate_threshold_youden(
    labels=calibration_df[LABEL_COLUMN],
    predictions=calibration_df[SCORE_COLUMN],
)
print(f"Calibrated threshold (Youden's J): {THRESHOLD:.4f}")

calibration_tpr, calibration_fpr = estimate_classifier_error_rates(
    labels=calibration_df[LABEL_COLUMN],
    predictions=calibration_df[SCORE_COLUMN],
    threshold=THRESHOLD,
)
print(f"\nCalibration set error rates at threshold={THRESHOLD:.4f}:")
print(f"  TPR (sensitivity): {calibration_tpr:.4f}")
print(f"  FPR (1-specificity): {calibration_fpr:.4f}")

# PACC parameters
PACC_POS_MEAN = calibration_df[calibration_df[LABEL_COLUMN] == 1][SCORE_COLUMN].mean()
PACC_NEG_MEAN = calibration_df[calibration_df[LABEL_COLUMN] == 0][SCORE_COLUMN].mean()
SOURCE_PREVALENCE = calibration_df[LABEL_COLUMN].mean()

print(f"\nPACC parameters (soft-score Rogan-Gladen):")
print(f"  E[h(X)|Y=1]: {PACC_POS_MEAN:.4f}")
print(f"  E[h(X)|Y=0]: {PACC_NEG_MEAN:.4f}")
print(f"\nSource prevalence for SLD: {SOURCE_PREVALENCE:.4f}")

# Verify
apparent_prev = (calibration_df[SCORE_COLUMN] >= THRESHOLD).mean()
true_prev = calibration_df[LABEL_COLUMN].mean()
print(f"\nVerification:")
print(f"  True prevalence in calibration set: {true_prev:.4f}")
print(f"  Apparent prevalence at threshold: {apparent_prev:.4f}")

In [ ]:
# Methods to compare — conditionally include MCGrad + Emb.
methods_to_plot = [
    'Raw Scores', 'Classify & Count', 'Rogan-Gladen', 'PACC',
    'SLD (EMQ)', 'Isotonic Regression', 'MCGrad',
]
if HAS_EMBEDDINGS:
    methods_to_plot.append('MCGrad + Emb.')

colors_map = {m: METHOD_COLORS[m] for m in methods_to_plot}

In [ ]:
# --- IPW baseline ---
# Standard covariate-shift method: estimate density ratio P(target)/P(source)
# via logistic regression, reweight source labels. Requires re-estimation per target.

from sklearn.linear_model import LogisticRegression

IPW_FEATURES = ['country', 'doc_type', 'decade']

def ipw_estimate(cal_df, target_df, label_col='law_crime', features=IPW_FEATURES):
    """Importance-weighted prevalence estimation.

    Trains logistic regression to distinguish source from target using features X,
    then reweights source labels by the density ratio. Standard covariate shift approach.
    Unlike MCGrad, this requires re-estimation for each new target population.
    """
    combined = pd.concat([
        cal_df[features].assign(_is_target=0),
        target_df[features].assign(_is_target=1),
    ], ignore_index=True)
    X = pd.get_dummies(combined[features], drop_first=True).values.astype(float)
    z = combined['_is_target'].values
    clf = LogisticRegression(max_iter=1000, random_state=42)
    clf.fit(X, z)
    n_cal = len(cal_df)
    p_target = clf.predict_proba(X[:n_cal])[:, 1]
    weights = p_target / np.maximum(1 - p_target, 1e-10)
    y = cal_df[label_col].values
    return np.clip(np.average(y, weights=weights), 0.0, 1.0)

# Add IPW to methods list
methods_to_plot.append('IPW')
colors_map['IPW'] = '#9467bd'  # purple

In [ ]:
# Compute prevalence estimates for each scenario along the shift gradient
all_results = {}

_emb_col = MCGRAD_EMB_COL if HAS_EMBEDDINGS else None

for scenario_label, info in scenarios.items():
    syn_df = info['df']
    estimates = compute_all_prevalence_estimates(
        target_df=syn_df,
        score_col=SCORE_COLUMN,
        ir_col=IR_COL,
        mcgrad_col=MCGRAD_COL,
        label_col=LABEL_COLUMN,
        cal_tpr=calibration_tpr,
        cal_fpr=calibration_fpr,
        pacc_pos_mean=PACC_POS_MEAN,
        pacc_neg_mean=PACC_NEG_MEAN,
        source_prevalence=SOURCE_PREVALENCE,
        threshold=THRESHOLD,
        mcgrad_emb_col=_emb_col,
    )
    all_results[scenario_label] = estimates
    clean_label = scenario_label.replace('\n', ' ')
    print(f"\n{clean_label}:")
    print(f"  True prevalence: {estimates['True Prevalence']:.4f}")
    display(compute_bias_table(estimates).round(4))

In [ ]:
# Add IPW estimates to all_results (computed separately because IPW
# uses calibration labels + target features, not h(X) scores)
print("Computing IPW estimates...")
for scenario_label, info in scenarios.items():
    target = info['df']
    ipw_est = ipw_estimate(calibration_df, target)
    all_results[scenario_label]['IPW'] = ipw_est
    true_prev = all_results[scenario_label]['True Prevalence']
    bias = (ipw_est - true_prev) * 100
    clean_label = scenario_label.replace('\n', ' ')
    print(f"  {clean_label}: IPW={ipw_est:.4f}, bias={bias:+.1f}pp")
print("Done.")

## 5. Bootstrap RMSE

200 bootstrap iterations per scenario. For each iteration, resample with
replacement from the scenario's source population and compute prevalence
estimates. Report bias and RMSE across resamples.

Bootstrap sample size: `min(len(source_population), 20_000)` to handle
sub-populations smaller than 20K (e.g., Belgium newspaper ~21K).

In [ ]:
print("Computing bootstrap RMSE (200 resamples per scenario)...")

_emb_col = MCGRAD_EMB_COL if HAS_EMBEDDINGS else None

bootstrap_config = {
    'Baseline\n(balanced test)': {
        'mode': 'uniform', 'source': test_df,
    },
    'Country shift\n(overweight Belgium)': {
        'mode': 'shift', 'source': test_df,
        'shift_fn': resample_with_country_shift,
        'shift_kwargs': {'target_country': 'Belgium', 'n_samples': N_SAMPLES},
    },
    'Doc-type shift\n(overweight bills)': {
        'mode': 'shift', 'source': test_df,
        'shift_fn': resample_with_doctype_shift,
        'shift_kwargs': {'target_doctype': 'bill', 'n_samples': N_SAMPLES},
    },
    'Party shift\n(right-heavy)': {
        'mode': 'shift', 'source': test_df,
        'shift_fn': resample_with_party_shift,
        'shift_kwargs': {'label_column': LABEL_COLUMN, 'shift': 'right_heavy', 'n_samples': N_SAMPLES},
    },
    'Spain media\n(OOD doc type)': {
        'mode': 'uniform', 'source': subpops['spain_media'],
    },
    'Belgium TV\n(OOD doc type)': {
        'mode': 'uniform', 'source': subpops['belgium_tv'],
    },
}

common_args = dict(
    score_col=SCORE_COLUMN, ir_col=IR_COL, mcgrad_col=MCGRAD_COL,
    label_col=LABEL_COLUMN, cal_tpr=calibration_tpr, cal_fpr=calibration_fpr,
    pacc_pos_mean=PACC_POS_MEAN, pacc_neg_mean=PACC_NEG_MEAN,
    source_prevalence=SOURCE_PREVALENCE, threshold=THRESHOLD,
    mcgrad_emb_col=_emb_col,
)

bootstrap_results = {}
for label, cfg in bootstrap_config.items():
    clean_label = label.replace('\n', ' ')
    print(f"  {clean_label}...")
    if cfg['mode'] == 'shift':
        bootstrap_results[label] = compute_bootstrap_rmse_shifted(
            cfg['source'],
            shift_fn=cfg['shift_fn'],
            shift_kwargs=cfg['shift_kwargs'],
            **common_args,
        )
    else:
        bootstrap_results[label] = compute_bootstrap_rmse(
            cfg['source'], **common_args,
        )

print("Done.")

In [ ]:
# IPW bootstrap (separate from main bootstrap since IPW uses calibration labels,
# not h(X) scores, and requires re-fitting a density ratio model per iteration)
print("Computing IPW bootstrap (200 resamples per scenario)...")

for label, cfg in bootstrap_config.items():
    clean_label = label.replace('\n', ' ')
    print(f"  {clean_label}...")
    ipw_biases = []
    for b in range(200):
        if cfg['mode'] == 'shift':
            target = cfg['shift_fn'](cfg['source'], random_state=b, **cfg['shift_kwargs'])
        else:
            source = cfg['source']
            target = source.sample(n=min(N_SAMPLES, len(source)), replace=True, random_state=b)
        true_prev = target[LABEL_COLUMN].mean()
        ipw_est = ipw_estimate(calibration_df, target)
        ipw_biases.append((ipw_est - true_prev) * 100)
    ipw_biases = np.array(ipw_biases)
    bootstrap_results[label]['IPW'] = {
        'bias': ipw_biases.mean(),
        'variance': ipw_biases.var(),
        'rmse': np.sqrt(np.mean(ipw_biases**2)),
    }
    print(f"    bias={ipw_biases.mean():+.2f}pp, rmse={np.sqrt(np.mean(ipw_biases**2)):.2f}pp")

print("Done.")

## 6. Results Table

Summary table showing bias and RMSE across the shift gradient,
mirroring the ACS Table 1 format. Rows ordered from no shift to maximum shift.

In [ ]:
# Summary table: bias and RMSE across shift gradient
shift_summary = []

scenario_order = list(scenarios.keys())

for scenario_label in scenario_order:
    estimates = all_results[scenario_label]
    bs = bootstrap_results[scenario_label]
    true_prev = estimates['True Prevalence']
    clean_label = scenario_label.replace('\n', ' ')
    shift_type = scenarios[scenario_label]['shift_type']
    row = {
        'Scenario': clean_label,
        'Shift Type': shift_type,
        'True Prevalence': f'{true_prev:.1%}',
    }
    for method in methods_to_plot:
        bias_pp = (estimates[method] - true_prev) * 100
        rmse_pp = bs[method]['rmse']
        row[f'{method} Bias'] = f'{bias_pp:+.2f}pp'
        row[f'{method} RMSE'] = f'{rmse_pp:.2f}pp'
    shift_summary.append(row)

summary_df = pd.DataFrame(shift_summary).set_index(['Scenario', 'Shift Type'])
print('=== Law & Crime Prevalence Estimation: Bias and RMSE Across Shift Gradient ===\n')
summary_df

## 7. Figure: Bias Across the Shift Gradient

Single-panel bar chart showing prevalence estimation bias for each method
across the shift gradient (from baseline to maximum shift).
Uses `METHOD_COLORS` from `plot_config.py`, same style as ACS Figure 2.

In [ ]:
scenario_labels = list(scenarios.keys())

fig, ax = plt.subplots(figsize=(14, 6.5))

x = np.arange(len(scenario_labels))
n_methods = len(methods_to_plot)
group_width = 0.82
bar_width = group_width / n_methods * 0.85

for i, method in enumerate(methods_to_plot):
    biases = [
        (all_results[s][method] - all_results[s]['True Prevalence']) * 100
        for s in scenario_labels
    ]
    offset = (i - n_methods / 2 + 0.5) * (group_width / n_methods)
    ax.bar(
        x + offset, biases, bar_width,
        label=method, color=colors_map[method],
        edgecolor='white', linewidth=0.5,
    )

ax.axhline(y=0, color='gray', linestyle='--', alpha=0.5)
ax.set_ylabel('Bias (percentage points)')
ax.set_xticks(x)
ax.set_xticklabels(scenario_labels, rotation=45, ha='right')
ax.legend(fontsize=7, loc='best')

# Add shift gradient annotation
ax.annotate(
    '', xy=(len(scenario_labels) - 0.5, ax.get_ylim()[0]),
    xytext=(-0.5, ax.get_ylim()[0]),
    arrowprops=dict(arrowstyle='->', color='#888888', lw=1.5),
)
ax.text(
    len(scenario_labels) / 2 - 0.5, ax.get_ylim()[0] * 0.95,
    'increasing distributional shift $\\longrightarrow$',
    ha='center', va='top', fontsize=9, color='#888888', style='italic',
)

fig.suptitle(
    'Law & Crime Prevalence Estimation Bias Across Shift Gradient',
    fontsize=13,
)
fig.tight_layout()
fig.savefig(
    '../paper/images/figure_cap_shift_gradient.png',
    dpi=300, bbox_inches='tight',
)
plt.show()